# CasDsl — a categorically organized CAS in Lean 4

This notebook walks through the CasDsl surface: a computer algebra system
whose operations are organized by the mathematical categories where they
first make sense. Every cell is ordinary Lean 4, elaborated by a persistent
worker. **Backend-blind syntax:** no expression ever names Sage, GAP, or an
algorithm — the routing layer selects implementations after the mathematical
operation is resolved.

We'll start with trusted arithmetic, then factorization, polynomials,
algebraic numbers, linear algebra, and calculus — and end with a deliberate
capability gap that shows what happens when a method is semantically
available but no backend has registered an implementation yet.

## 1 · Trusted arithmetic and assertions

`assert` is an operational assertion in the ordinary CAS sense: the
predicate is computed and trusted. Only `true` lets the cell commit — a
false or unknown result is a cell error, and the notebook state rolls back.
No Lean theorem is generated; this is a CAS, not a proof obligation
machine.

We start with the simplest possible assertion — integer arithmetic — then
move to modular arithmetic.

In [1]:
assert 2 + 3 = 5

1:0: ✓ 2 + 3 = 5


The `in ℤ/5` suffix changes the ring the assertion is evaluated in.
`2 + 3` is 5 in ℤ, but 5 ≡ 0 in ℤ/5, so the assertion holds.

In [2]:
assert 2 + 3 = 0 in ℤ/5

1:0: ✓ 2 + 3 = 0 in ℤ/5


## 2 · Factorization

`factor` is declared on the category of factorization-domain elements.
An integer receives it because `EuclideanElems(ℤ) ≤ FactorizationElems(ℤ)`
— the method arrives by functor composition, not by leaf-specific
forwarding code. The computation is routed to Sage, but the expression
`n.factor()` never names it.

We bind `n := 360` as an integer, then ask for its factorization.

In [3]:
let n := 360 in ℤ

1:0: n := 360 ∈ ℤ


In [4]:
n.factor()

1:0: 2^3 * 3^2 * 5


2^3 * 3^2 * 5

`gcd` works the same way — it's a method on the category of GCD-domain
elements, and integers inherit it through the same subcategory chain.

In [5]:
assert gcd(84, 30) = 6

1:0: ✓ gcd(84, 30) = 6


## 3 · Polynomials

Polynomial rings are first-class categories. We'll define
$p(x) = x^3 - 2x + 1$ in $\mathbb{Z}[x]$ and explore what the system can
tell us about it.

The `let p(x) := … in ℤ[x]` syntax binds `p` as a polynomial over the
integers. The `(x)` after the name tells the parser this is a polynomial
binder — `x` becomes the indeterminate.

In [6]:
let p(x) := x^3 - 2x + 1 in ℤ[x]

1:0: p := x^3 - 2x + 1 ∈ ℤ[x]


`p.deg()` returns the degree. This is a category-owned method — it's
declared on the category of polynomials and inherited by every concrete
polynomial ring.

In [7]:
p.deg()

1:0: 3


3

`p.roots()` asks for the roots of $p$ *in its coefficient ring*.
Since $p \in \mathbb{Z}[x]$, this asks for integer roots.

$x^3 - 2x + 1$ has no integer roots — its three roots are $1$,
$\frac{-1 + \sqrt{5}}{2}$, and $\frac{-1 - \sqrt{5}}{2}$, and only $1$
is rational. So the empty set `{}` is the **correct answer**, not a
failure. This is a deliberate design choice: an empty result is an answer,
not an error.

In [8]:
p.roots()

1:0: {}


{}

To get the rational root, we move $p$ to $\mathbb{Q}[x]$ using the
canonical inclusion $\mathbb{Z} \subseteq \mathbb{Q}$. The `map … to …`
syntax uses the registered canonical map between the two rings — it lifts
each coefficient from ℤ to ℚ.

The result is a new polynomial $q$ over ℚ with the same coefficients,
now interpreted as rational numbers.

In [9]:
map p to ℚ[x]

1:0: q := x^3 - 2x + 1 ∈ ℚ[x]


Now we can evaluate $q$ at a point. `q(1)` substitutes $x = 1$ and
returns $1^3 - 2 \cdot 1 + 1 = 0$, confirming that $1$ is a root.

In [10]:
q(1)

1:0: 0


0

## 4 · Exact algebraic numbers

CasDsl works with exact algebraic numbers — never decimals unless you
explicitly request an approximation. The ⊆-chain
$\mathbb{N} \subseteq \mathbb{Z} \subseteq \mathbb{Q} \subseteq \mathbb{R} \subseteq \mathbb{C}$
is read off the canonical-map registry, so membership and transport
between these systems is automatic.

We'll bind $z = 2 + 2i$ in ℂ and verify its absolute value.

In [11]:
let z := 2 + 2i in ℂ

1:0: z := 2 + 2i ∈ ℂ


$|2 + 2i| = \sqrt{2^2 + 2^2} = \sqrt{8} = 2\sqrt{2}$. The system
returns the exact surd, not $2.828\ldots$.

In [12]:
assert |z| = 2√2

1:0: ✓ |z| = 2√2


Numerical approximation is an operation **on** an exact value, not a
replacement for it. `map √2 to ℝ/O(1/10^{10})` asks for $\sqrt{2}$
approximated to within $10^{-10}$ in ℝ. The tolerance is a request, not a
quotient — the underlying value remains exact.

In [13]:
map √2 to ℝ/O(1/10^{10})

1:0: 1.4142135623… ∈ ℝ/O(1/10^{10})


1.4142135623…

## 5 · Linear algebra

Exact matrix arithmetic over ℚ. We'll define a $2 \times 2$ matrix,
compute its determinant, and invert it — all exactly, with rational
entries.

In [14]:
let M := [[1, 2], [3, 4]] in Mat₂(ℚ)

1:0: M := [[1, 2], [3, 4]] ∈ Mat₂(ℚ)


$\det(M) = 1 \cdot 4 - 2 \cdot 3 = -2$. The result is exact — no
floating-point.

In [15]:
M.det()

1:0: -2


-2

$M^{-1} = \frac{1}{-2} \begin{bmatrix} 4 & -2 \\ -3 & 1 \end{bmatrix}
= \begin{bmatrix} -2 & 1 \\ 3/2 & -1/2 \end{bmatrix}$. Again, exact
rational entries.

In [16]:
M⁻¹

1:0: [[-2, 1], [3/2, -1/2]]


[[-2, 1], [3/2, -1/2]]

## 6 · Calculus

CasDsl distinguishes the **universal differential** $d(f)$ — a 1-form —
from the **derivation** $(d/dx)(f)$ — a polynomial. They are different
types and not equal, even when their coefficients match.

The indefinite integral $\int f\,dx$ returns the **coset** of
antiderivatives: $x^3 + x^2/2 + \mathbb{Q}$ means "the set of all
functions of the form $x^3 + x^2/2 + c$ where $c \in \mathbb{Q}$."

We'll work with $f(x) = 3x^2 + x$ over ℚ.

In [17]:
let f(x) := 3x^2 + x in ℚ[x]

1:0: f := 3x^2 + x ∈ ℚ[x]


$d(f) = (6x + 1)\,dx$ — the universal differential, a 1-form. The $dx$
is part of the value; it's not just notation.

In [18]:
d(f)

1:0: d(f) = (6x + 1) dx


(6x + 1) dx

$(d/dx)(f) = 6x + 1$ — the derivation, a plain polynomial. Note the
absence of $dx$: this is the coefficient of the differential, not the
differential itself.

In [19]:
(d/dx)(f)

1:0: (d/dx)(f) = 6x + 1


6x + 1

$\int f\,dx = x^3 + x^2/2 + \mathbb{Q}$. The $+ \mathbb{Q}$ is the
constant of integration, presented as a coset: any rational constant added
to $x^3 + x^2/2$ is also an antiderivative. This is not a notational
convention — the system models the indefinite integral as a set.

In [20]:
∫ f dx

1:0: x^3 + x^2/2 + ℚ


x^3 + x^2/2 + ℚ

## 7 · Documented ceiling

Not every mathematically meaningful operation has a backend implementation
yet. When you ask for one, the system returns a **structured capability
gap** — it names the operation, the receiver, and the reason it can't
proceed. This is not a crash or a hidden method; it's an auditable entry
in the developer backlog.

Here, `det` is semantically available on $\text{Mat}_2(\mathbb{Z}/5)$
(a matrix ring over a commutative ring always has a determinant), but no
backend has registered a realization for matrices over $\mathbb{Z}/5$
yet. Over ℚ it works (we just used it); over ℤ/5 it's a gap.

In [21]:
let N := [[1, 2], [3, 4]] in Mat₂(ℤ/5)
N.det()

1:0: NoImplementation: det is semantically available on Mat₂(ℤ/5) but no backend realization is registered
